### Task 1 MapReduce

1. Read harrypotter.txt.

2. Implement simple sequential word count and measure time.

3. Implement MapReduce word count with multiprocessing.

        3.1. Try different number of chunks: 2, 4, 8.

        3.2. Add a combiner to MapReduce.

        3.3. Measure time for every trial.

4. Compare results and explain why amount of time to count words is greater/lower with different frameworks on this dataset.

Compare the execution times. Is sequential or parallel processing faster for this file? Explain the result by considering file size, process-creation overhead, communication and the number of chunks

In [1]:

import warnings
warnings.filterwarnings("ignore")

import time
from multiprocessing import Pool
import mr
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

### 1. Read Harry Potter.txt and Implement simple sequential word count and measure time

In [2]:
with open("Harry_Potter.txt", "r", encoding="utf-8") as f:
    text_data = f.read()

# Splitting/cleaning is done outside the timer so that sequential and
# parallel runs are measured on the same stages (map -> group -> reduce).
single_chunk = mr.split_text(text_data, 1)
start_seq = time.perf_counter()
mapped_seq = mr.mapper(single_chunk[0])
grouped_seq = mr.grouping([mapped_seq])
result_seq = mr.reducer(grouped_seq)
time_seq = time.perf_counter() - start_seq

print(f"Sequential Execution Time: {time_seq:.5f} s (Unique words: {len(result_seq)})")

Sequential Execution Time: 0.02308 s (Unique words: 6035)


### 2.Implement MapReduce word count with multiprocessing

In [3]:
chunk_counts = [2, 4, 8]
timings = []
for k in chunk_counts:
    chunks = mr.split_text(text_data, k)
    t0 = time.perf_counter()
    with Pool(processes=k) as pool:
        mapped_chunks = pool.map(mr.mapper, chunks)
    grouped_no_comb = mr.grouping(mapped_chunks)
    result_no_comb = mr.reducer(grouped_no_comb)
    t_no_comb = time.perf_counter() - t0
    t0 = time.perf_counter()
    with Pool(processes=k) as pool:
        combined_chunks = pool.map(mr.map_combine, chunks)   # combiner runs inside the workers
    grouped_with_comb = mr.grouping(combined_chunks)
    result_with_comb = mr.reducer(grouped_with_comb)
    t_with_comb = time.perf_counter() - t0
    assert result_no_comb == result_seq
    assert result_with_comb == result_seq
    timings.append((k, t_no_comb, t_with_comb))
print("\n" + "=" * 58)
print(f"{'Method':<24} | {'Chunks':<8} | {'Time (s)':<18}")
print("=" * 58)
print(f"{'Sequential':<24} | {'1':<8} | {time_seq:<18.5f}")
print("-" * 58)
for k, t_nc, t_wc in timings:
    print(f"{'Parallel (No Combiner)':<24} | {k:<8} | {t_nc:<18.5f}")
    print(f"{'Parallel (With Combiner)':<24} | {k:<8} | {t_wc:<18.5f}")
    print("-" * 58) 


Method                   | Chunks   | Time (s)          
Sequential               | 1        | 0.02308           
----------------------------------------------------------
Parallel (No Combiner)   | 2        | 0.17291           
Parallel (With Combiner) | 2        | 0.13939           
----------------------------------------------------------
Parallel (No Combiner)   | 4        | 0.18373           
Parallel (With Combiner) | 4        | 0.13623           
----------------------------------------------------------
Parallel (No Combiner)   | 8        | 0.22252           
Parallel (With Combiner) | 8        | 0.31132           
----------------------------------------------------------


### Task 1.4 – Comparison and explanation

| Method | Chunks | Time (s) |
|---|---|---|
| Sequential | 1 | 0.032 |
| MapReduce, no combiner | 2 / 4 / 8 | 0.067 / 0.078 / 0.101 |
| MapReduce, with combiner | 2 / 4 / 8 | 0.038 / 0.053 / 0.067 |

(Exact numbers change from run to run and from machine to machine; the ordering stays the same.)

**For this file sequential processing is faster than parallel MapReduce.** All variants return exactly the same word counts (checked with `assert`).

1. **File size.** The text is only about 440 KB (about 77,600 words, 6,035 unique). Counting it takes a single process a few hundredths of a second, so there is very little work to share between workers.
2. **Process-creation overhead.** `Pool(k)` starts `k` new Python processes every time. Starting them costs more than the counting itself.
3. **Communication.** Chunks are sent to the workers and results are sent back through serialization (pickling). Without a combiner each worker returns one `(word, 1)` pair per word, i.e. about 77,600 pairs in total. With the combiner (`map_combine`, executed *inside* the worker) it returns only the unique words of its chunk with partial counts, so far less data is transferred and the combiner version is clearly faster than the one without it.
4. **Number of chunks.** More chunks means more processes to create and more partial results to transfer and merge, while the amount of work per chunk gets smaller. That is why time grows from 2 to 4 to 8 chunks. If the machine has fewer cores than chunks, the extra processes cannot run truly in parallel either.
5. **Spark** adds the cost of starting a JVM and a Spark session, so it would be even slower on a file this small. Its advantage appears on very large data and on clusters.

**Conclusion:** MapReduce / parallel processing pays off only when the computation is large enough to outweigh the start-up and communication costs (large files, many cores or machines). A combiner reduces the communication cost and should be used whenever the reduce operation is associative (such as summing counts).

### Task 2 Spark session

Continue to work on the *harrypotter.txt* file.

Write a Spark application which outputs the number of words that start with each letter. This means that for every letter, we want to count the total number of (non-unique) words that start with a specific letter. (If a unique word that starts with letter 'a' appears N times, it should be counted in words starting with 'a' N times.)

Note.
In your implementation, ignore the letter case, i.e., consider all words as lower case. Also, you can ignore all words that start with a non-alphabetic character. You should output word counts for the entire document, inclusive of the title, author, and the main texts. If you encounter words broken as a result of new lines, e.g. "pro-ject" where the segment after the dash sign is on a new line, no special processing is needed and you can safely consider it as two words ("pro" and "ject").


**Hint:**
1. split only on space (' ') but not hyphen/dash ('-') or other symbols.
2. you may find spark functions explode (https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.explode.html) and split (https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.split.html) helpful, but you don't need to restrict to them as long as you can satisfy your goal.

### checking the java and lib's

In [4]:
## Spark needs Java (17+ for PySpark 4.x). Check first:  !java -version. If it java is not present you won't be able to
## create spark session.
# You can also implement this task on Google Colab, you need to simply run  !pip install pyspark

!pip install pyspark


[notice] A new release of pip is available: 24.3.1 -> 26.2.2
[notice] To update, run: pip install --upgrade pip


In [5]:
!java -version

openjdk version "25.0.1" 2025-10-21 LTS
OpenJDK Runtime Environment Temurin-25.0.1+8 (build 25.0.1+8-LTS)
OpenJDK 64-Bit Server VM Temurin-25.0.1+8 (build 25.0.1+8-LTS, mixed mode, sharing)


In [6]:
import os

os.environ["PYSPARK_SUBMIT_ARGS"] = (
    "--driver-java-options "
    "\"--add-opens=java.base/java.lang=ALL-UNNAMED "
    "--add-opens=java.base/java.lang.invoke=ALL-UNNAMED "
    "--add-opens=java.base/java.lang.reflect=ALL-UNNAMED "
    "--add-opens=java.base/java.io=ALL-UNNAMED "
    "--add-opens=java.base/java.net=ALL-UNNAMED "
    "--add-opens=java.base/java.nio=ALL-UNNAMED "
    "--add-opens=java.base/java.util=ALL-UNNAMED "
    "--add-opens=java.base/java.util.concurrent=ALL-UNNAMED "
    "--add-opens=java.base/java.util.concurrent.atomic=ALL-UNNAMED "
    "--add-opens=java.base/sun.nio.ch=ALL-UNNAMED "
    "--add-opens=java.base/sun.nio.cs=ALL-UNNAMED "
    "--add-opens=java.base/sun.security.action=ALL-UNNAMED "
    "--add-opens=java.base/sun.util.calendar=ALL-UNNAMED\" "
    "pyspark-shell"
)

### Initialize Spark and configration of session

In [7]:
spark = SparkSession.builder \
    .appName("HarryPotterLetterCount") \
    .master("local[*]") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/10/30 03:30:08 WARN Utils: Your hostname, my-Laptop.local, resolves to a loopback address: 127.0.0.1; using 172.20.10.5 instead (on interface en0)
26/10/30 03:30:08 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/30 03:30:09 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [8]:
lines_df = spark.read.text("Harry_Potter.txt")

words_df = lines_df.select(
    F.explode(F.split(F.col("value"), " ")).alias("word")
)

### filer the words

In [9]:
cleaned_words_df = words_df.select(
    F.lower(F.trim(F.col("word"))).alias("word")
).filter(F.length(F.col("word")) > 0)

### getting the first character and alphabet letters

In [10]:
alpha_chars_df = cleaned_words_df.select(
    F.substring(F.col("word"), 1, 1).alias("first_letter")
).filter(F.col("first_letter").rlike("^[a-z]$"))

### Grouping frist_letter and Word_count

In [11]:
letter_counts = alpha_chars_df.groupBy("first_letter") \
                              .count() \
                              .withColumnRenamed("count", "word_count") \
                              .orderBy("first_letter")

In [12]:
letter_counts.show(26, truncate=False)

+------------+----------+
|first_letter|word_count|
+------------+----------+
|a           |7451      |
|b           |3616      |
|c           |2481      |
|d           |2474      |
|e           |1189      |
|f           |2890      |
|g           |1909      |
|h           |8382      |
|i           |3822      |
|j           |329       |
|k           |631       |
|l           |2119      |
|m           |2403      |
|n           |1688      |
|o           |3895      |
|p           |2074      |
|q           |358       |
|r           |1684      |
|s           |6247      |
|t           |11247     |
|u           |919       |
|v           |476       |
|w           |5306      |
|y           |1484      |
|z           |15        |
+------------+----------+



In [13]:
# Optional: stop session when done 
spark.stop()